### 1. 사용 내역 데이터 전처리

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
base_path = "/content/drive/MyDrive/26-1_BITAmin_TS_project"

### 1.1 데이터 로딩 및 1차 전처리

In [3]:
import os
import glob
import pandas as pd
import numpy as np

In [10]:
USAGE_DIR = os.path.join(base_path, "raw_data/usage")
REPAIR_DIR = os.path.join(base_path, "raw_data/repair")

usage_files = sorted(glob.glob(os.path.join(USAGE_DIR, "*.csv")))
repair_files = sorted(glob.glob(os.path.join(REPAIR_DIR, "*.csv")))

print("usage 파일 수:", len(usage_files))
print("repair 파일 수:", len(repair_files))

usage 파일 수: 59
repair 파일 수: 12


In [12]:
# 누락 컬럼 확인
keep_cols = ["자전거번호", "반납일시", "반납대여소번호"]

for file in usage_files:

    # 인코딩 fallback
    try:
        df = pd.read_csv(file, encoding="cp949", low_memory=False, nrows=5)
    except UnicodeDecodeError:
        try:
            df = pd.read_csv(file, encoding="utf-8-sig", low_memory=False, nrows=5)
        except UnicodeDecodeError:
            df = pd.read_csv(file, encoding="utf-8", low_memory=False, nrows=5)

    df.columns = df.columns.str.strip()

    rename_map = {
        "반납 대여소번호": "반납대여소번호",
        "반납 대여소 번호": "반납대여소번호",
    }
    df = df.rename(columns=rename_map)

    missing_cols = [col for col in keep_cols if col not in df.columns]
    if missing_cols:
        print("파일:", os.path.basename(file))
        print("누락 컬럼:", missing_cols)
        print("실제 컬럼:", df.columns.tolist())
        print("-" * 80)

In [13]:
# usage 데이터 전처리
keep_cols = ["자전거번호", "반납일시", "반납대여소번호"]

summary_list = []

for i, file in enumerate(usage_files):
    print(f"[{i+1}/{len(usage_files)}] 처리 중:", os.path.basename(file))

    df = pd.read_csv(file, encoding="cp949", low_memory=False)
    df.columns = df.columns.str.strip()

    # 컬럼명 통일
    df = df.rename(columns={
        "반납 대여소번호": "반납대여소번호",
        "반납 대여소 번호": "반납대여소번호",
    })

    # 누락 컬럼 보정
    for c in keep_cols:
        if c not in df.columns:
            df[c] = pd.NA

    df_cut = df[keep_cols].copy()

    # 데이터 타입 정리
    df_cut["자전거번호"] = df_cut["자전거번호"].astype(str).str.strip().str.upper()
    df_cut["반납일시"] = pd.to_datetime(df_cut["반납일시"], errors="coerce")
    df_cut["반납대여소번호"] = (
        df_cut["반납대여소번호"].astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^0-9]", "", regex=True)
        .replace({"": pd.NA})
    )

    # 핵심 컬럼 결측치 제거
    df_cut = df_cut.dropna(subset=["자전거번호", "반납일시", "반납대여소번호"])

    # 전처리 된 각 데이터 합치기
    summary_list.append(df_cut)

[1/59] 처리 중: 2102.csv
[2/59] 처리 중: 2103.csv
[3/59] 처리 중: 2104.csv
[4/59] 처리 중: 2105.csv
[5/59] 처리 중: 2106.csv
[6/59] 처리 중: 2107.csv
[7/59] 처리 중: 2108.csv
[8/59] 처리 중: 2109.csv
[9/59] 처리 중: 2110.csv
[10/59] 처리 중: 2111.csv
[11/59] 처리 중: 2112.csv
[12/59] 처리 중: 2201.csv
[13/59] 처리 중: 2202.csv
[14/59] 처리 중: 2203.csv
[15/59] 처리 중: 2204.csv
[16/59] 처리 중: 2205.csv
[17/59] 처리 중: 2206.csv
[18/59] 처리 중: 2207.csv
[19/59] 처리 중: 2208.csv
[20/59] 처리 중: 2209.csv
[21/59] 처리 중: 2210.csv
[22/59] 처리 중: 2211.csv
[23/59] 처리 중: 2212.csv
[24/59] 처리 중: 2301.csv
[25/59] 처리 중: 2302.csv
[26/59] 처리 중: 2303.csv
[27/59] 처리 중: 2304.csv
[28/59] 처리 중: 2305.csv
[29/59] 처리 중: 2306.csv
[30/59] 처리 중: 2307.csv
[31/59] 처리 중: 2308.csv
[32/59] 처리 중: 2309.csv
[33/59] 처리 중: 2310.csv
[34/59] 처리 중: 2311.csv
[35/59] 처리 중: 2312.csv
[36/59] 처리 중: 2401.csv
[37/59] 처리 중: 2402.csv
[38/59] 처리 중: 2403.csv
[39/59] 처리 중: 2404.csv
[40/59] 처리 중: 2405.csv
[41/59] 처리 중: 2406.csv
[42/59] 처리 중: 2407.csv
[43/59] 처리 중: 2408.csv
[44/59] 처리 중: 2409.c

In [15]:
# 전체 데이터 합치기
usage_all = pd.concat(summary_list, ignore_index=True)

# display(usage_all)
print("전체 shape:", usage_all.shape)

# 저장
usage_all.to_parquet(f"{base_path}/processed_data/usage_all.parquet", index=False)
print(f"저장 완료: {base_path}/processed_data/usage_all.parquet")

전체 shape: (198668499, 3)
저장 완료: /content/drive/MyDrive/26-1_BITAmin_TS_project/processed_data/usage_all.parquet


In [25]:
display(usage_all.head())

,자전거번호,반납일시,반납대여소번호
0,SPB-51323,2021-02-01 00:03:05,01554
1,SPB-52771,2021-02-01 00:04:59,01981
2,SPB-33780,2021-02-01 00:05:10,01360
3,SPB-52936,2021-02-01 00:05:27,02639
4,SPB-41150,2021-02-01 00:06:10,01708


In [16]:
# 결측치 확인
usage_all.isna().sum()

,0
자전거번호,0
반납일시,0
반납대여소번호,0


In [17]:
%pip install pyarrow

### 2. 고장 내역 데이터 전처리

### 2.1 데이터 로딩 및 전처리

In [18]:
# repair 데이터 샘플 확인
repair_sample = pd.read_csv(repair_files[0], encoding="cp949", low_memory=False) if repair_files else pd.DataFrame()
print(repair_sample.shape)
print(repair_sample.columns.tolist())
repair_sample.head()

(1408, 3)
['자전거번호', '등록일시', '고장구분']


,자전거번호,등록일시,고장구분
0,SPB-41936,2021-01-01 00:44:00,기타
1,SPB-42181,2021-01-01 02:42:00,타이어
2,SPB-36237,2021-01-01 03:53:00,기타
3,SPB-33399,2021-01-01 04:56:00,체인
4,SPB-36328,2021-01-01 08:21:00,기타


In [20]:
repair_files = sorted(glob.glob(os.path.join(REPAIR_DIR, "*.csv")))
repair_files

['/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2101.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2102_2106.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2107_2112.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2201_2206.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2207_2212.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2301_2306.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2307_2310.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2311_2312.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2401_2406.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2407_2412.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2501_2506.csv',
 '/content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2507_2512.csv']

In [21]:
# 각 데이터 파일별로 컬럼명 확인
# 인코딩 언어 확인
for file in repair_files:
    try:
        df_temp = pd.read_csv(file, encoding='cp949', nrows=3)
    except:
        try:
            df_temp = pd.read_csv(file, encoding='utf-8', nrows=3)
        except:
            df_temp = pd.read_excel(file, nrows=3)

    df_temp.columns = df_temp.columns.str.strip()
    print(f"\n파일명: {file}")
    print(df_temp.columns.tolist())


파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2101.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2102_2106.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2107_2112.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2201_2206.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2207_2212.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2301_2306.csv
['자전거번호', '등록일시', '구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2307_2310.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2311_2312.csv
['자전거번호', '등록일시', '고장구분']

파일명: /content/drive/MyDrive/26-1_BITAmin_TS_project/raw_data/repair/2401_2406.csv
['자전거번호', '등록일시', '구분']

파일명: /content/drive/MyDrive

In [22]:
# 컬럼 통일하고 데이터 concat
repair_dfs = []

for file in repair_files:
    print(f"처리 중: {os.path.basename(file)}")

    try:
        df = pd.read_csv(file, encoding="cp949", low_memory=False)
    except:
        df = pd.read_csv(file, encoding="utf-8", low_memory=False)

    # 컬럼명 공백 제거
    df.columns = df.columns.str.strip()

    # 컬럼명 통일
    if '구분' in df.columns and '고장구분' not in df.columns:
        df = df.rename(columns={'구분': '고장구분'})

    # 필요한 컬럼만 남기기
    df = df[['자전거번호', '등록일시', '고장구분']].copy()

    # 등록일시 문자열 정리
    df['등록일시'] = df['등록일시'].astype(str).str.strip()
    df['등록일시'] = (
        df['등록일시']
        .str.replace('.', '-', regex=False)
        .str.replace('/', '-', regex=False)
    )
    df['등록일시'] = df['등록일시'].replace(['', 'nan', 'None'], pd.NA)

    # mixed 포맷 datetime 변환
    df['등록일시'] = pd.to_datetime(
        df['등록일시'],
        format='mixed',
        errors='coerce'
    )

    # 변환 실패 확인
    na_count = df['등록일시'].isna().sum()
    # print(f"등록일시 결측치: {na_count}")

    repair_dfs.append(df)

repair_all = pd.concat(repair_dfs, ignore_index=True)

print("최종 shape:", repair_all.shape)
print("최종 결측 수:", repair_all['등록일시'].isna().sum())
display(repair_all)

처리 중: 2101.csv
처리 중: 2102_2106.csv
처리 중: 2107_2112.csv
처리 중: 2201_2206.csv
처리 중: 2207_2212.csv
처리 중: 2301_2306.csv
처리 중: 2307_2310.csv
처리 중: 2311_2312.csv
처리 중: 2401_2406.csv
처리 중: 2407_2412.csv
처리 중: 2501_2506.csv
처리 중: 2507_2512.csv
최종 shape: (779052, 3)
최종 결측 수: 0


,자전거번호,등록일시,고장구분
0,SPB-41936,2021-01-01 00:44:00,기타
1,SPB-42181,2021-01-01 02:42:00,타이어
2,SPB-36237,2021-01-01 03:53:00,기타
3,SPB-33399,2021-01-01 04:56:00,체인
4,SPB-36328,2021-01-01 08:21:00,기타
...,...,...,...
779047,SPB-68754,2025-12-31 23:12:55,체인
779048,SPB-62680,2025-12-31 23:32:06,기타
779049,SPB-34168,2025-12-31 23:33:45,체인
779050,SPB-34168,2025-12-31 23:33:45,기타


In [24]:
repair_all.to_parquet(f"{base_path}/processed_data/repair_all.parquet", index=False)
print(f"저장 완료: {base_path}/processed_data/repair_all.parquet")

저장 완료: /content/drive/MyDrive/26-1_BITAmin_TS_project/processed_data/repair_all.parquet


### 3. 위치 데이터 전처리

In [ ]:
# location 데이터 전처리
LOCATION_PATH = os.path.join(base_path, "raw_data/location.csv")

loc = pd.read_csv(
    LOCATION_PATH,
    encoding="utf-8-sig",
    header=None,
    skiprows=5,                         # 실제 데이터 시작 행
    usecols=[0, 4, 5],                  # 대여소번호, 위도, 경도
    names=["대여소 번호", "위도", "경도"],
    dtype={0: "string"}
)

# 타입/포맷 정리
loc["대여소 번호"] = (
    loc["대여소 번호"].astype("string").str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^0-9]", "", regex=True)
    .replace({"": pd.NA})
)
loc["위도"] = pd.to_numeric(loc["위도"], errors="coerce")
loc["경도"] = pd.to_numeric(loc["경도"], errors="coerce")

# 유효 좌표 + 결측/중복 제거
valid_coord = loc["위도"].between(33, 39) & loc["경도"].between(124, 132)
location_all = (
    loc[valid_coord]
    .dropna(subset=["대여소 번호", "위도", "경도"])
    .drop_duplicates("대여소 번호")
    .reset_index(drop=True)
)

print("shape:", location_all.shape)
display(location_all)

location_all.to_parquet(f"{base_path}/processed_data/location_all.parquet", index=False)

shape: (2799, 3)


,대여소 번호,위도,경도
0,00102,37.555649,126.910629
1,00103,37.554951,126.910835
2,00104,37.550629,126.914986
3,00105,37.550007,126.914825
4,00106,37.548645,126.912826
...,...,...,...
2794,06185,37.573410,126.843452
2795,06187,37.555347,126.820724
2796,06188,37.556190,126.864639
2797,06189,37.564484,126.848305
